# AC 209b / CS 1090b MS4 Main Notebook - Project 66

**Team:** Harry Hu, Tom Shan, Wendy Wang, Kemeng Zhang

This notebook implements the MS4 workflow for testing whether emotion-informed features improve author-level prediction of the four binary MBTI dimensions over corrected text-only baselines. The default path is safe: it uses synthetic data and compact-artifact checks only, so it does not download the full Reddit dataset or start neural training.

## Runtime Flags

The notebook has three distinct modes:

- `SMOKE_TEST=True`: run synthetic end-to-end checks for preprocessing, splitting, weighting, metrics, and model construction.
- `RUN_REAL_DATA_SMOKE=True`: optionally verify real KaggleHub/Hugging Face access on tiny samples. This may require network access and `uv sync --extra full --extra dev`.
- `RUN_FULL_TRAINING=True`: blocked by default. Enable only when intentionally launching long local MPS jobs.

In [ ]:
from pathlib import Path
import sys

RUN_FULL_TRAINING = False
RUN_REAL_DATA_SMOKE = False
SMOKE_TEST = True
REDDIT_SAMPLE_NROWS = 1000

candidate = Path.cwd()
if not (candidate / "src" / "ms4mbti").exists():
    repo_relative = candidate / "milestones" / "ms4_final_modeling_deliverables" / "code"
    if repo_relative.exists():
        candidate = repo_relative

CODE_DIR = candidate.resolve()
SRC_DIR = CODE_DIR / "src"
if not (SRC_DIR / "ms4mbti").exists():
    raise RuntimeError(f"Could not find MS4 source package under {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"CODE_DIR={CODE_DIR}")
print(f"RUN_FULL_TRAINING={RUN_FULL_TRAINING}")
print(f"RUN_REAL_DATA_SMOKE={RUN_REAL_DATA_SMOKE}")
print(f"SMOKE_TEST={SMOKE_TEST}")

## Environment Check

Default notebook execution should work on CPU or MPS. Full training is the only path that requires MPS.

In [ ]:
import importlib.metadata as metadata
import pandas as pd
from IPython.display import display

from ms4mbti.config import DIMENSION_SPECS, PROJECT_NUMBER, TARGET_COLUMNS, TEAM_MEMBERS, RunConfig
from ms4mbti.training import require_mps_for_full_training, set_global_seed, torch_environment

config = RunConfig(run_full_training=RUN_FULL_TRAINING, smoke_test=SMOKE_TEST)
set_global_seed(config.seed)
require_mps_for_full_training(config.run_full_training)

packages = ["ms4mbti", "numpy", "pandas", "scikit-learn", "torch", "nbformat"]
versions = []
for package in packages:
    try:
        versions.append({"package": package, "version": metadata.version(package)})
    except metadata.PackageNotFoundError:
        versions.append({"package": package, "version": "not installed"})

print(f"Project {PROJECT_NUMBER}")
print("Team:", ", ".join(TEAM_MEMBERS))
print("Target columns:", TARGET_COLUMNS)
print("Torch environment:", torch_environment())
display(pd.DataFrame(versions))
display(pd.DataFrame([
    {
        "dimension": spec.display,
        "target_col": spec.target_col,
        "positive_label": spec.positive_label,
        "negative_label": spec.negative_label,
    }
    for spec in DIMENSION_SPECS.values()
]))

## Project Framing

MS3 showed that vanilla text-only and text-plus-emotion GRU baselines collapsed to majority-class author predictions on `E/I`, `N/S`, and `J/P`, with only weak minority signal on `F/T`. MS4 therefore tests a corrected protocol: class-weighted BCE, reduced prolific-author dominance, soft author aggregation, and validation-tuned thresholds. The critical comparison is fixed text-only GRU versus fixed GRU plus emotion probabilities under the same preprocessing, split, loss recipe, aggregation, and metrics.

## Data Access

Production data access should use public APIs, not project-local raw CSVs:

- Reddit MBTI target data: KaggleHub dataset `minhaozhang1/reddit-mbti-dataset`, file `reddit_post.csv`.
- Emotion source data: Hugging Face dataset `AdamCodd/emotion-balanced`.

The real-data smoke path is optional and intentionally small. The full Reddit CSV should only be resolved in the long-running experiment path.

In [ ]:
from ms4mbti.data import emotion_dataset_overview, load_reddit_sample_via_kagglehub

if RUN_REAL_DATA_SMOKE:
    reddit_sample = load_reddit_sample_via_kagglehub(config, nrows=REDDIT_SAMPLE_NROWS)
    print("Reddit API sample shape:", reddit_sample.shape)
    display(reddit_sample.head())

    emotion_overview = emotion_dataset_overview(config, max_rows_per_split=3)
    for split_name, info in emotion_overview.items():
        print(f"Emotion split={split_name}, rows={info['n_rows']}, columns={info['columns']}")
        display(info["preview"])
else:
    print("RUN_REAL_DATA_SMOKE=False. Skipping network/API smoke checks.")
    print("To verify public data access later: uv run --extra full --extra dev jupyter lab, then set RUN_REAL_DATA_SMOKE=True.")

## Default Modeling Frame

Until the full Reddit run is launched, this notebook uses a deterministic synthetic Reddit-like corpus. It is not evidence for the report; it is a quality gate that exercises the same code paths the real data will use.

In [ ]:
from ms4mbti.quality import make_synthetic_reddit
from ms4mbti.preprocessing import (
    add_masked_text,
    audit_mbti_leakage,
    author_label_conflicts,
    clean_reddit_frame,
    resolve_author_label_conflicts,
)
from ms4mbti.data import apply_ms3_filters

smoke_config = RunConfig(
    min_words=5,
    min_posts_per_author=3,
    max_posts_per_author=5,
    stage2_max_length=24,
    train_size=0.625,
    val_size=0.1875,
    test_size=0.1875,
    run_full_training=False,
    smoke_test=True,
)

raw_posts = make_synthetic_reddit(seed=smoke_config.seed)
cleaned_posts = clean_reddit_frame(raw_posts)
conflicts = author_label_conflicts(cleaned_posts)
resolved_posts = resolve_author_label_conflicts(cleaned_posts, strategy="drop")
leakage_before = audit_mbti_leakage(resolved_posts, text_col="text")
masked_posts = add_masked_text(resolved_posts)
leakage_after = audit_mbti_leakage(masked_posts, text_col="text_masked")
modeling_posts = apply_ms3_filters(
    masked_posts,
    min_words=smoke_config.min_words,
    min_posts_per_author=smoke_config.min_posts_per_author,
    max_posts_per_author=smoke_config.max_posts_per_author,
    seed=smoke_config.seed,
)

print("Raw posts:", len(raw_posts))
print("Cleaned posts:", len(cleaned_posts))
print("Conflicted authors dropped:", len(conflicts))
print("Modeling posts after MS3 filters:", len(modeling_posts))
display(pd.DataFrame([leakage_before, leakage_after], index=["before_masking", "after_masking"]))
display(modeling_posts.head())

## Preprocessing Audits

The same audits should be reported on the real MS4 corpus: explicit MBTI term leakage, author-label consistency, MS3 filtering retention, and author-level split balance.

In [ ]:
from ms4mbti.split import attach_splits, make_author_frame, split_authors, split_balance_table

original_authors = resolved_posts["author"].nunique()
modeling_authors = modeling_posts["author"].nunique()
author_frame = make_author_frame(modeling_posts)
split_result = split_authors(
    author_frame,
    train_size=smoke_config.train_size,
    val_size=smoke_config.val_size,
    test_size=smoke_config.test_size,
    seed=smoke_config.seed,
)
posts_with_splits = attach_splits(modeling_posts, split_result.authors)
split_balance = split_balance_table(split_result.authors)

print("Authors before filters:", original_authors)
print("Authors after filters:", modeling_authors)
print("Split method:", split_result.method)
print("Split warnings:", split_result.warnings or "none")
display(split_result.authors["split"].value_counts().rename_axis("split").reset_index(name="n_authors"))
display(split_balance)

In [ ]:
from ms4mbti.token_audit import token_truncation_audit

class_balance = []
for target in TARGET_COLUMNS:
    class_balance.append({
        "target": target,
        "positive_authors": int(split_result.authors[target].sum()),
        "negative_authors": int((1 - split_result.authors[target]).sum()),
        "positive_share": float(split_result.authors[target].mean()),
    })
class_balance = pd.DataFrame(class_balance)

post_cap_summary = (
    posts_with_splits.groupby("author")
    .size()
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .rename("retained_posts_per_author")
    .reset_index()
)

truncation = token_truncation_audit(
    posts_with_splits,
    max_length=smoke_config.stage2_max_length,
    target_cols=TARGET_COLUMNS,
)

print("Author-level class balance")
display(class_balance)
print("Retained posts per author summary")
display(post_cap_summary)
print("Token truncation post summary")
display(pd.DataFrame([truncation["post_summary"]]))
print("Token truncation author summary")
display(pd.DataFrame([truncation["author_summary"]]))
print("Token truncation by split")
display(truncation["by_split"])

## Audit Visualizations

These figures are intentionally generated in the default path so the notebook has report-ready visual structure before long training starts. In the final run, the same plotting calls should be reused on the full masked corpus.

In [ ]:
from ms4mbti.viz import (
    plot_class_balance,
    plot_leakage_audit,
    plot_posts_per_author,
    plot_split_balance,
    plot_token_truncation_by_split,
    set_plot_style,
)

set_plot_style()

leakage_plot_df = pd.DataFrame([
    {"stage": "before masking", **leakage_before},
    {"stage": "after masking", **leakage_after},
])
posts_per_author = posts_with_splits.groupby("author").size()

fig_leakage = plot_leakage_audit(leakage_plot_df)
fig_class_balance = plot_class_balance(class_balance)
fig_posts_per_author = plot_posts_per_author(posts_per_author)
fig_split_balance = plot_split_balance(split_balance)
fig_truncation = plot_token_truncation_by_split(truncation["by_split"])

for fig in [fig_leakage, fig_class_balance, fig_posts_per_author, fig_split_balance, fig_truncation]:
    display(fig)


## Weighting Recipe

For real experiments, select the class-weight variant on the fixed text-only GRU validation set only, then lock that recipe for all fixed GRU models. This avoids choosing weights based on the final emotion model.

In [ ]:
from ms4mbti.weighting import author_post_loss_weights, compute_pos_weights, effective_label_distribution

train_posts = posts_with_splits.loc[posts_with_splits["split"] == "train"].copy()
train_posts["post_loss_weight"] = author_post_loss_weights(train_posts)

effective_distribution = effective_label_distribution(
    train_posts,
    sample_weight=train_posts["post_loss_weight"],
)
pos_weight_inverse = compute_pos_weights(
    train_posts,
    variant="inverse",
    sample_weight=train_posts["post_loss_weight"],
)
pos_weight_sqrt = compute_pos_weights(
    train_posts,
    variant="sqrt",
    sample_weight=train_posts["post_loss_weight"],
)

print("Effective train distribution after per-author post weighting")
display(effective_distribution)
display(pd.concat([pos_weight_inverse, pos_weight_sqrt], axis=1))

## Model Comparison Plan

Historical MS3 rows should be reported separately if they are not rerun on the masked MS4 corpus. The main MS4 comparison table should use the same masked corpus, author split, metric pipeline, and validation-threshold protocol.

In [ ]:
from ms4mbti.experiments import model_comparison_plan

comparison_plan = model_comparison_plan()
display(comparison_plan)

## Stage 1 Emotion Classifier Plan

Stage 1 is a source-task model, not final evidence by itself. Its job is to produce six soft emotion probabilities for Reddit posts. The final claim depends on downstream author-level MBTI metrics against the fixed text-only control.

In [ ]:
stage1_plan = pd.DataFrame([
    {
        "step": "source data",
        "implementation": "Load AdamCodd/emotion-balanced through Hugging Face datasets",
        "default_notebook_status": "optional smoke only",
        "full_run_output": "train/validation/test row counts and label distribution",
    },
    {
        "step": "model",
        "implementation": "DistilBERT sequence classifier, max length 64",
        "default_notebook_status": "not trained",
        "full_run_output": "validation/test accuracy, macro-F1, confusion matrix",
    },
    {
        "step": "Reddit inference",
        "implementation": "Batched inference over masked, filtered Reddit posts",
        "default_notebook_status": "not run",
        "full_run_output": "post-level six-probability cache with metadata",
    },
    {
        "step": "diagnostics",
        "implementation": "Compare source emotion distribution against Reddit predicted emotion distribution",
        "default_notebook_status": "placeholder contract",
        "full_run_output": "source-vs-Reddit emotion distribution figure",
    },
])
display(stage1_plan)

stage1_metric_contract = pd.DataFrame([
    {"metric": "accuracy", "split": "validation/test", "reason": "basic source-task quality check"},
    {"metric": "macro_f1", "split": "validation/test", "reason": "guards against class-specific weakness"},
    {"metric": "confusion_matrix", "split": "test", "reason": "shows which emotions transfer poorly before Reddit inference"},
    {"metric": "reddit_mean_probability", "split": "full masked Reddit corpus", "reason": "checks source-target emotion distribution shift"},
])
display(stage1_metric_contract)


## Baseline and Metric Skeleton

The default run computes majority and TF-IDF author baselines on synthetic data. In the real run, this section should produce the first non-neural result table before any GRU training starts.

In [ ]:
from ms4mbti.baselines import train_linear_author_baseline
from ms4mbti.evaluation import (
    evaluate_with_validation_thresholds,
    majority_baseline_author_scores,
    metric_table,
)

train_authors = split_result.authors.loc[split_result.authors["split"] == "train"].copy()
majority_scores = majority_baseline_author_scores(train_authors, split_result.authors)
majority_score_cols = tuple(f"score_majority_{target}" for target in TARGET_COLUMNS)
majority_test_metrics = metric_table(
    majority_scores.loc[majority_scores["split"] == "test"],
    score_cols=majority_score_cols,
    thresholds={target: 0.5 for target in TARGET_COLUMNS},
)
majority_test_metrics.insert(0, "model_id", "majority")

linear_scores, linear_models = train_linear_author_baseline(posts_with_splits, seed=smoke_config.seed)
linear_score_cols = tuple(f"score_linear_{target}" for target in TARGET_COLUMNS)
linear_eval = evaluate_with_validation_thresholds(linear_scores, score_cols=linear_score_cols)
linear_test_metrics = linear_eval["test_metrics"].copy()
linear_test_metrics.insert(0, "model_id", "linear_tfidf")

baseline_metrics = pd.concat([majority_test_metrics, linear_test_metrics], ignore_index=True)
display(baseline_metrics)

## Baseline Visualization

This plot verifies that the metric-table shape supports final report figures. On real data, the same function should compare the linear baseline against the fixed GRU rows.

In [ ]:
from ms4mbti.viz import plot_metric_comparison

fig_baseline_balacc = plot_metric_comparison(baseline_metrics, metric="balanced_accuracy")
fig_baseline_f1 = plot_metric_comparison(baseline_metrics, metric="f1")
display(fig_baseline_balacc)
display(fig_baseline_f1)


## Main Results Contract

The final results section should keep historical MS3 diagnosis separate from the MS4 controlled comparison. The table below defines the schema expected from compact artifacts and final reports.

In [ ]:
main_results_schema = pd.DataFrame([
    {"column": "model_id", "required": True, "description": "majority, linear_tfidf, fixed_gru_text_only, fixed_gru_rnn_emotion, fixed_gru_distilbert_emotion"},
    {"column": "split", "required": True, "description": "validation or test"},
    {"column": "target", "required": True, "description": "target_E, target_S, target_T, target_J"},
    {"column": "threshold", "required": True, "description": "validation-tuned decision threshold, copied unchanged to test"},
    {"column": "balanced_accuracy", "required": True, "description": "primary thresholded metric"},
    {"column": "f1", "required": True, "description": "secondary thresholded metric"},
    {"column": "minority_precision", "required": True, "description": "precision for configured positive/minority side"},
    {"column": "minority_recall", "required": True, "description": "recall for configured positive/minority side"},
    {"column": "average_precision", "required": True, "description": "PR-AUC / AP from continuous author scores"},
    {"column": "roc_auc", "required": True, "description": "ROC-AUC from continuous author scores"},
    {"column": "raw_accuracy", "required": False, "description": "secondary only; not a headline ranking metric"},
])
display(main_results_schema)

historical_vs_controlled = pd.DataFrame([
    {"table": "historical_ms3_diagnosis", "allowed_rows": "MS3 B1/B2 and majority floor", "warning": "label unmasked/uncontrolled if not rerun"},
    {"table": "main_ms4_controlled", "allowed_rows": "linear_tfidf, fixed text-only GRU, fixed RNN-emotion GRU, fixed DistilBERT-emotion GRU", "warning": "same masked corpus, split, metric pipeline only"},
])
display(historical_vs_controlled)


## Default Figure Export

The default smoke figures are exported locally so the report/video workflow has a concrete file handoff. These smoke figures are not final results; final figures should be regenerated from the full run or compact artifacts.

In [ ]:
from ms4mbti.viz import save_figures

figure_dir = config.artifact_dir / "figures_smoke"
figure_paths = save_figures(
    [
        fig_leakage,
        fig_class_balance,
        fig_posts_per_author,
        fig_split_balance,
        fig_truncation,
        fig_baseline_balacc,
        fig_baseline_f1,
    ],
    figure_dir,
    prefix="smoke",
)
print("Exported smoke figures:")
for path in figure_paths:
    print("-", path)


## Stage 2 GRU Skeleton

This section prepares the same arrays and model objects the real fixed GRU experiments will use, but it does not train. Full training remains behind `RUN_FULL_TRAINING`.

In [ ]:
from ms4mbti.models import count_parameters
from ms4mbti.stage2 import build_stage2_from_arrays, make_torch_dataset, prepare_stage2_split_arrays

posts_for_stage2 = posts_with_splits.copy()
posts_for_stage2["post_loss_weight"] = 1.0
posts_for_stage2.loc[train_posts.index, "post_loss_weight"] = train_posts["post_loss_weight"]

stage2_arrays_by_split, stage2_vocab = prepare_stage2_split_arrays(
    posts_for_stage2,
    max_length=smoke_config.stage2_max_length,
    sample_weight_col="post_loss_weight",
    vocab_max_size=1000,
    vocab_min_freq=1,
)
train_dataset = make_torch_dataset(stage2_arrays_by_split["train"])
stage2_model = build_stage2_from_arrays(
    stage2_arrays_by_split["train"],
    vocab_size=stage2_vocab.size if stage2_vocab is not None else 30000,
)

print("Stage 2 vocab size:", stage2_vocab.size if stage2_vocab is not None else "tokenizer-based")
print("Train dataset rows:", len(train_dataset))
print("Stage 2 model parameter count:", count_parameters(stage2_model))
print("No Stage 2 training has been launched.")

## Validation-Only Recipe Selection Placeholder

When fixed text-only GRU validation metrics exist for inverse-frequency and square-root class weights, pass those two validation metric tables into `select_weight_recipe`. Do not use test metrics for this choice.

In [ ]:
from ms4mbti.experiments import select_weight_recipe

# Placeholder demonstration using duplicated baseline metrics. Replace with fixed text-only GRU
# validation metrics for inverse and sqrt variants after real training.
recipe_demo = select_weight_recipe({
    "inverse": linear_eval["validation_metrics"],
    "sqrt": linear_eval["validation_metrics"],
})
print("Recipe-selection API demo selected:", recipe_demo.selected_variant)
display(recipe_demo.summary)

## Compact Artifact Contract

The submitted notebook should regenerate final tables from compact author-level artifacts. Large post-level caches and checkpoints should be documented but excluded from the zip unless course staff explicitly requests them.

In [ ]:
from ms4mbti.cache import base_compact_metadata, compact_artifact_contract, write_manifest

contract = compact_artifact_contract()
display(contract)

label_encoding = {
    key: {
        "target_col": spec.target_col,
        "positive_label": spec.positive_label,
        "negative_label": spec.negative_label,
    }
    for key, spec in DIMENSION_SPECS.items()
}
example_metadata = base_compact_metadata(
    artifact_type="author_scores",
    created_by="cs1090b_ms4_main_group66.ipynb",
    split_id="masked_author_split_seed_209066",
    masking_status="explicit_mbti_terms_masked",
    label_encoding=label_encoding,
    threshold_objective="balanced_accuracy",
    extra={"model_id": "fixed_gru_text_only", "stage2_max_length": config.stage2_max_length},
)
print("Example compact artifact metadata keys:")
display(pd.DataFrame([example_metadata]))

## Figure Export Policy

For the final submission, export only the polished figures used by the report/video into the report or video workspace. Keep exploratory notebook figures reproducible from compact artifacts instead of submitting large intermediate image dumps.

## Interpretation, Limitations, and Broader Impact

These points should be filled with final numbers after the controlled run, but the interpretive frame is fixed now to avoid post-hoc storytelling.

In [ ]:
interpretation_frame = pd.DataFrame([
    {"case": "emotion models beat fixed text-only", "interpretation": "Emotion probabilities add incremental author-level signal after imbalance and aggregation fixes."},
    {"case": "emotion models tie or lose to fixed text-only", "interpretation": "Targeted MS4 fixes may recover minority signal, but transferred emotion probabilities add limited value beyond text."},
    {"case": "linear TF-IDF beats GRU family", "interpretation": "Lexical author-level aggregation is a strong baseline; report honestly and treat neural results as not automatically superior."},
    {"case": "N/S remains weak", "interpretation": "Severe skew and label noise may dominate even under weighting and threshold tuning."},
])
display(interpretation_frame)

limitations = pd.DataFrame([
    {"limitation": "Self-reported MBTI labels", "consequence": "Labels are noisy and may cap achievable performance."},
    {"limitation": "MBTI construct validity", "consequence": "Results should not be framed as clinical or definitive personality assessment."},
    {"limitation": "Source-target domain shift", "consequence": "Emotion model trained on short source texts may miscalibrate Reddit emotion probabilities."},
    {"limitation": "No subreddit/time metadata", "consequence": "Topic and community effects cannot be separated from author style."},
    {"limitation": "Public text privacy", "consequence": "Report aggregate patterns only; do not identify or profile individual Reddit users."},
])
display(limitations)

broader_impact = pd.DataFrame([
    {"area": "Potential benefit", "point": "A controlled negative or positive result can clarify when interpretable emotion features help text classification."},
    {"area": "Misuse risk", "point": "Personality inference from public writing can be overinterpreted or used for profiling."},
    {"area": "Mitigation", "point": "Keep evaluation aggregate, emphasize uncertainty, and avoid claims about individual psychological truth."},
])
display(broader_impact)


## References and Disclosure

The final report should cite datasets, libraries, model papers, and AI assistance. Keep exact URLs in the report references section.

In [ ]:
references = pd.DataFrame([
    {"type": "dataset", "name": "Reddit MBTI dataset", "citation_or_url": "https://www.kaggle.com/datasets/minhaozhang1/reddit-mbti-dataset/data"},
    {"type": "dataset", "name": "AdamCodd/emotion-balanced", "citation_or_url": "https://huggingface.co/datasets/AdamCodd/emotion-balanced"},
    {"type": "library", "name": "PyTorch MPS backend", "citation_or_url": "https://docs.pytorch.org/docs/stable/notes/mps"},
    {"type": "library", "name": "PyTorch BCEWithLogitsLoss", "citation_or_url": "https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html"},
    {"type": "library", "name": "Hugging Face Transformers sequence classification", "citation_or_url": "https://huggingface.co/docs/transformers/tasks/sequence_classification"},
    {"type": "library", "name": "Hugging Face Datasets loading", "citation_or_url": "https://huggingface.co/docs/datasets/loading"},
    {"type": "library", "name": "scikit-learn balanced accuracy", "citation_or_url": "https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html"},
    {"type": "paper", "name": "DistilBERT", "citation_or_url": "https://arxiv.org/abs/1910.01108"},
    {"type": "tooling", "name": "Generative AI assistance", "citation_or_url": "Used for code drafting, repository organization, and notebook scaffolding; final claims should be verified by the team."},
])
display(references)


## Full Training Launch Plan

After this notebook's default path remains green, the long run should proceed in this order:

1. Run real-data smoke checks.
2. Build the masked full modeling corpus and author split.
3. Train/fine-tune Stage 1 DistilBERT emotion model and cache Reddit emotion probabilities.
4. Train fixed text-only GRU with inverse and square-root `pos_weight` variants on validation only.
5. Lock one weight recipe from fixed text-only validation mean balanced accuracy.
6. Train fixed RNN-emotion and DistilBERT-emotion GRU models with the locked recipe.
7. Tune thresholds on validation authors and evaluate test authors once.
8. Save compact author-level artifacts and manifests.

In [ ]:
if RUN_FULL_TRAINING:
    raise RuntimeError(
        "Full training is deliberately blocked in this notebook checkpoint. "
        "Turn this guard into explicit launch cells only after the real-data smoke path and compact artifact contract are accepted."
    )
print("Notebook completed without downloading full data or launching training.")